# cnn-2d-v5b-weighted-sampler — 2D EfficientNet-B0, clean-only + class-balanced sampler

**Condition:** 2D spectrogram CNN, clean 4,800-row training set, class-balanced `WeightedRandomSampler`.

**Base:** `cnn-2d-v5-clean-data.ipynb`.

**Only change:** `sampler = get_sampler(train_df, label_col='label')` (from `src.data`) and the train
`DataLoader` uses `sampler=sampler` (`shuffle=True` dropped — they are mutually exclusive). This mirrors how
`cnn-1d-v7b-weighted-sampler.ipynb` differs from `cnn-1d-v7-clean-data.ipynb`. A sanity assert checks that
`train_df['label']` equals the argmax of the normalized votes that `SpectrogramDataset` uses for its hard labels.

Validation is the shared clean test split (1,139 rows); all other settings are identical to v5.


In [ ]:
# cnn-2d-v5b-weighted-sampler: 2D CNN on spectrograms — EfficientNet-B0 + KL loss + early stopping
# condition: 2D, clean-only (4,800 rows) + class-balanced WeightedRandomSampler
!pip install timm -q

In [ ]:
import sys, os, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

# ── src imports from uploaded code dataset ──────────────────────────────────
IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    CODE_DIR = '/kaggle/input/datasets/xiaosufrankhu/hms-eeg-code'
else:
    CODE_DIR = os.path.abspath('../kaggle_upload')

sys.path.insert(0, CODE_DIR)
sys.path.insert(0, os.path.join(CODE_DIR, 'src'))

from src.config_2d  import Config2D
from src.dataset_2d import SpectrogramDataset
from src.model_2d   import EfficientNetEEG

print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'} | torch {torch.__version__}")

In [ ]:
cfg = Config2D()

if IS_KAGGLE:
    DATA_ROOT = '/kaggle/input/competitions/hms-harmful-brain-activity-classification'
    # train_test_split.csv (not train.csv) — contains inner_fold and split columns
    cfg.metadata_csv    = '/kaggle/input/datasets/xiaosufrankhu/hms-eeg-code/train_test_split.csv'
    cfg.spectrogram_dir = os.path.join(DATA_ROOT, 'train_spectrograms')
else:
    DATA_ROOT = os.path.abspath('../')
    cfg.metadata_csv    = os.path.abspath('../data_meta_splits/train_test_split.csv')
    cfg.spectrogram_dir = os.path.join(DATA_ROOT, 'train_spectrograms')

# ── reproducibility ─────────────────────────────────────────────────────────
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
torch.cuda.manual_seed_all(cfg.seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE  = torch.device(cfg.device)
USE_AMP = DEVICE.type == 'cuda'

print(cfg)
print(f"Device: {DEVICE} | AMP: {USE_AMP}")

In [ ]:
# Convert parquet → npy once; subsequent runs skip existing files.
from src.dataset_2d import preprocess_spectrograms
preprocess_spectrograms(cfg.spectrogram_dir, cfg.spec_cache_dir, n_jobs=-1)

In [ ]:
from src.data import load_clean_data, get_sampler

if IS_KAGGLE:
    CLEAN_PATH = '/kaggle/input/datasets/xiaosufrankhu/hms-eeg-code/train_clean.csv'
else:
    CLEAN_PATH = os.path.abspath('../data_meta_splits/train_clean.csv')
val_df = load_clean_data(CLEAN_PATH, split='test')  # 1,139 rows — same held-out clean test set used by v4 and all 1D notebooks

train_df = load_clean_data(CLEAN_PATH, split='trainval')  # 4,800 rows — clean-only training set (v3 used the full noisy trainval split)
train_ds = SpectrogramDataset(train_df, cfg.spectrogram_dir, cfg)
val_ds   = SpectrogramDataset(val_df, cfg.spectrogram_dir, cfg)

# Sanity: the 'label' column get_sampler() reads must equal the hard labels
# SpectrogramDataset derives (argmax of the normalized vote columns).
assert np.array_equal(train_df['label'].to_numpy().astype(int), train_ds.hard_labels), \
    "train_df['label'] != argmax(normalized votes) used by SpectrogramDataset"

# class-balanced sampling — the ONLY difference from v5.
# sampler and shuffle=True are mutually exclusive, so shuffle is dropped.
sampler = get_sampler(train_df, label_col='label')

train_loader = DataLoader(
    train_ds, batch_size=cfg.batch_size, sampler=sampler,
    num_workers=cfg.num_workers, pin_memory=True,
)
val_loader = DataLoader(
    val_ds, batch_size=cfg.batch_size, shuffle=False,
    num_workers=cfg.num_workers, pin_memory=True,
)

print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,}")

# batch shape sanity check
batch = next(iter(train_loader))
print(f"image      : {tuple(batch['image'].shape)}")
print(f"soft_label : {tuple(batch['soft_label'].shape)}")
print(f"label      : {tuple(batch['label'].shape)}")

In [ ]:
model = EfficientNetEEG(
    backbone    = cfg.backbone,
    num_classes = cfg.num_classes,
    pretrained  = cfg.pretrained,
).to(DEVICE)

total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Backbone    : {cfg.backbone}")
print(f"Parameters  — total: {total_params:,} | trainable: {trainable_params:,}")

In [ ]:
# ========== SHARED EVALUATION SETUP ==========
# Same code path as the 1D CNN notebooks (src/evaluation.validate) so val
# numbers are computed the same way everywhere. It divides by
# len(loader.dataset) (not len(loader)), so an uneven last batch no longer
# biases val KL.
# NOTE: SpectrogramDataset's batch dict uses "image"/"label"/"soft_label"
# (not EEGDatasetV2's "x"/"y"/"soft_y"), so x_key/y_key/soft_key must be
# passed explicitly — soft_key="soft_y" would KeyError, there is no such
# key in this dataset.
from src.evaluation import validate
from src.losses import build_loss

eval_cfg = {"loss": {"name": "kl", "label_smoothing": 0.0},
            "model": {"num_classes": 6}}
# NOTE: must be the KLLoss from build_loss() — it applies log_softmax itself.
# Passing a bare nn.KLDivLoss() here would skip that and give wrong numbers.
val_loss_fn = build_loss(eval_cfg)

In [ ]:
# v6: 2D spectrogram CNN + KL loss + cosine LR + early stopping
GRAD_CLIP = 1.0

criterion = nn.KLDivLoss(reduction='batchmean')   # training loss only
optimizer = AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = CosineAnnealingLR(optimizer, T_max=cfg.num_epochs)
scaler    = torch.amp.GradScaler('cuda', enabled=USE_AMP)

best_kl, best_epoch, best_f1_at_best_kl, history = float('inf'), 0, 0.0, []
wait = 0

for epoch in range(1, cfg.num_epochs + 1):
    # ── train ────────────────────────────────────────────────────────────────
    model.train()
    t0         = time.time()
    train_loss = 0.0

    for batch in train_loader:
        x    = batch['image'].to(DEVICE)       # (B, 4, H, W)
        soft = batch['soft_label'].to(DEVICE)  # (B, 6)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            logits    = model(x)
            log_probs = F.log_softmax(logits, dim=1)
            loss      = criterion(log_probs, soft)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    scheduler.step()

    # ── validate (shared: src/evaluation.validate) ──────────────────────────
    val_loss, metrics, val_probs, val_y_true = validate(
        model, val_loader, val_loss_fn, eval_cfg, DEVICE,
        soft_key='soft_label', x_key='image', y_key='label',
    )
    avg_kl    = val_loss
    macro_f1  = metrics['macro_f1']
    avg_train = train_loss / len(train_loader)
    elapsed   = time.time() - t0

    history.append({
        'epoch': epoch, 'train_kl': avg_train,
        'val_kl': avg_kl, 'macro_f1': macro_f1,
    })

    print(f"Epoch {epoch:03d} | train_kl {avg_train:.4f} | "
          f"val_kl {avg_kl:.4f} | "
          f"macro_f1 {macro_f1:.4f} | {elapsed:.0f}s")

    # checkpoint / early-stop on val_kl (the headline metric), not macro_f1
    if avg_kl < best_kl:
        best_kl, best_epoch, best_f1_at_best_kl, wait = avg_kl, epoch, macro_f1, 0
        torch.save(model.state_dict(), 'best_model_v5b.pt')
        print(f"  ✓ saved best model (kl={best_kl:.4f})")
    else:
        wait += 1
        if wait >= cfg.patience:
            print(f"Early stopping at epoch {epoch} (no improvement for {cfg.patience} epochs)")
            break

print(f"\nTraining complete. Best epoch {best_epoch}: "
      f"val_kl={best_kl:.4f}, macro_f1={best_f1_at_best_kl:.4f}")

In [ ]:
epochs   = [h['epoch']    for h in history]
train_kl = [h['train_kl'] for h in history]
val_kl   = [h['val_kl']   for h in history]
macro_f1 = [h['macro_f1'] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, train_kl, label='train KL')
ax1.plot(epochs, val_kl,   label='val KL')
ax1.axvline(best_epoch, color='red', linestyle='--', label=f'best epoch={best_epoch}')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('KL Divergence')
ax1.set_title('KL Divergence (train vs val)'); ax1.legend()

ax2.plot(epochs, macro_f1, color='green')
ax2.axvline(best_epoch, color='red', linestyle='--', label=f'macro_f1={best_f1_at_best_kl:.3f}')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Macro F1')
ax2.set_title('Validation Macro F1'); ax2.legend()

plt.tight_layout()
plt.savefig('training_curves_v5b.png', dpi=150)
plt.show()


In [ ]:
# ========== FINAL SUMMARY ==========
NOTEBOOK_NAME = "cnn-2d-v5b-weighted-sampler.ipynb"

# (a) per-epoch log table
print(f"{'epoch':>5} | {'train_kl':>9} | {'val_kl':>9} | {'macro_f1':>9}")
print("-" * 43)
for h in history:
    print(f"{h['epoch']:>5d} | {h['train_kl']:>9.4f} | {h['val_kl']:>9.4f} | {h['macro_f1']:>9.4f}")

# (b) anchor 1: best-val_kl epoch — the checkpointed model
_best_kl = min(history, key=lambda h: h["val_kl"])
print(f"\nAnchor 1 (best val_kl, checkpointed): epoch {_best_kl['epoch']} | "
      f"val_kl={_best_kl['val_kl']:.4f} | macro_f1={_best_kl['macro_f1']:.4f}")

# (c) anchor 2: best-macro_f1 epoch — read from history, does not affect checkpointing
_best_f1 = max(history, key=lambda h: h["macro_f1"])
print(f"Anchor 2 (best macro_f1)            : epoch {_best_f1['epoch']} | "
      f"val_kl={_best_f1['val_kl']:.4f} | macro_f1={_best_f1['macro_f1']:.4f}")

# (d) final epoch, and drift of val_kl from the best-KL epoch to the last epoch
_last = history[-1]
_pct  = (_last["val_kl"] - _best_kl["val_kl"]) / _best_kl["val_kl"] * 100
print(f"Final epoch                         : epoch {_last['epoch']} | "
      f"val_kl={_last['val_kl']:.4f} | macro_f1={_last['macro_f1']:.4f}")
print(f"val_kl change, best-KL epoch -> last epoch: {_pct:+.1f}%")

# (e) one-line summary — copy into notebooks/RESULTS.md
print(f"\n{NOTEBOOK_NAME} | "
      f"val_kl={_best_kl['val_kl']:.4f} | "
      f"val_f1={_best_kl['macro_f1']:.4f} | "
      f"epoch={_best_kl['epoch']}")
